# Demo 2 JAX Sharding Report: Batch-Axis Data Sharding for ViT Inference

This report explains how Demo 2 uses JAX sharding in the ViT inference workflow and how the retrieved TPU artifacts should be interpreted.

The main claim is concise:

> Demo 2 uses explicit batch-axis data sharding. The image batch is placed over a one-dimensional `data` mesh, the jitted inference step declares explicit input/output shardings, and the artifacts record both the resolved layout and the timing results.

The main evidence is inference-only: single-device `v6e-1` artifacts and sharded multi-device `v6e-8` artifacts for Imagenette 320 validation manifests. The fine-tuning script is mentioned only to show that the same batch-axis convention is also implemented for training-shaped inputs.

## 1. JAX Sharding Concepts Used in This Report

JAX sharding describes how a logical global `jax.Array` is laid out across physical devices. The program can still be written as a global-array computation, while JAX/XLA uses the declared layout to place arrays and partition computation.

The report uses these concepts:

| Concept | Role in this report |
| --- | --- |
| `Mesh` | A named grid of visible JAX devices. Demo 2 uses a one-dimensional mesh named `data`. |
| `PartitionSpec` | A rule mapping logical array axes to mesh axes. |
| `NamedSharding` | The concrete runtime sharding object: `Mesh + PartitionSpec`. |
| `jax.device_put` | Places an array onto a specific sharding layout. |
| `jax.jit(in_shardings=..., out_shardings=...)` | Compiles a function while declaring input/output layout contracts. |

For a ViT input batch shaped `[batch, channels, height, width]`, Demo 2 uses:

```python
PartitionSpec("data", None, None, None)
```

This means that the leading batch axis is split across the `data` mesh, while each image tensor remains intact on its assigned shard. For logits shaped `[batch, num_classes]`, the corresponding layout is:

```python
PartitionSpec("data", None)
```

The ViT parameters are not explicitly sharded in this report. The implemented feature is input-batch sharding for inference, with matching batch-input support in the fine-tuning script.

## 2. Where Sharding Appears in Demo 2

The sharding usage can be summarized by three operations. The relevant implementation files are `src/jax_tpu_project/sharding.py` and `examples/pretrained_vit_inference.py`, but the report only needs the core sharding pattern.

### 2.1 Create the layout

```python
mesh = Mesh(devices, ("data",))

image_sharding = NamedSharding(
    mesh,
    PartitionSpec("data", None, None, None),
)
logits_sharding = NamedSharding(
    mesh,
    PartitionSpec("data", None),
)
```

This is the point where the global batch dimension is mapped to a physical device mesh.

### 2.2 Place image batches on that layout

```python
image_batch = jax.device_put(image_batch, image_sharding)
```

The array is still used as a JAX array, but its leading batch dimension now has a concrete multi-device layout.

### 2.3 Declare the layout at the `jit` boundary

```python
inference_step = jax.jit(
    inference_step_impl,
    in_shardings=(None, image_sharding),
    out_shardings=logits_sharding,
)
```

The first argument is the model parameters, so `None` means they are not explicitly sharded here. The second argument is the image batch, so it receives the batch-axis sharding. This is the central place where JAX sharding enters the compiled inference step.

## 3. Batch-Axis Layout and Per-Device Work

The sharded artifacts use an eight-device `v6e-8` TPU runtime. Since the image batch is split across the `data` mesh, the useful work per device depends on the global batch size:

| Global batch size | Visible devices | Per-device batch size |
| ---: | ---: | ---: |
| 8 | 8 | 1 |
| 16 | 8 | 2 |
| 64 | 8 | 8 |
| 256 | 8 | 32 |
| 1024 | 8 | 128 |

This table is the key to interpreting the timing results. At small global batches, each device receives very little work. At larger global batches, the local batch on each device is large enough to better amortize the overhead of using the multi-device layout.

In [ ]:
# Setup: read-only helpers for artifact summaries.

from __future__ import annotations

import json
import math
import re
from pathlib import Path
from typing import Any

from IPython.display import Markdown, display


def find_repo_root(start: Path) -> Path:
    """Find the repository root from the current notebook/kernel path."""
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    return start


REPO_ROOT = find_repo_root(Path.cwd())
ARTIFACT_DIRS = [
    REPO_ROOT / "runs" / "vit-inference",
    Path.cwd() / "runs" / "vit-inference",
    Path.cwd().parent / "runs" / "vit-inference",
    Path.cwd(),
    Path.cwd().parent,
]


ARTIFACT_PATTERN = re.compile(
    r"demo2_cloud_imagenette320_"
    r"(?P<dataset>val256|valfull)_tpu_"
    r"(?P<family>single_v6e1|sharded_v6e8)_b"
    r"(?P<batch>\d+)\.json$"
)


EXPECTED_ARTIFACTS = [
    f"demo2_cloud_imagenette320_val256_tpu_{family}_b{batch}.json"
    for family in ("single_v6e1", "sharded_v6e8")
    for batch in (8, 16, 64, 256)
] + [
    f"demo2_cloud_imagenette320_valfull_tpu_{family}_b{batch}.json"
    for family in ("single_v6e1", "sharded_v6e8")
    for batch in (8, 16, 64, 256, 1024)
]


def find_artifact(name: str) -> Path | None:
    for directory in ARTIFACT_DIRS:
        candidate = directory / name
        if candidate.exists():
            return candidate
    return None


def clean_mesh(mesh_shape: Any) -> str:
    if isinstance(mesh_shape, dict) and mesh_shape:
        return ", ".join(f"{axis}={count}" for axis, count in mesh_shape.items())
    return "no explicit mesh"


def clean_jit_level(level: Any) -> str:
    if level in (None, "none"):
        return "unsharded jit"
    return str(level)


def clean_fallback(value: Any) -> str:
    return "None" if value in (None, "") else str(value)


def sharding_label(family: str) -> str:
    return "single v6e-1" if family == "single_v6e1" else "sharded v6e-8"


def load_artifact_records() -> tuple[list[dict[str, Any]], list[str]]:
    records: list[dict[str, Any]] = []
    missing: list[str] = []

    for name in EXPECTED_ARTIFACTS:
        path = find_artifact(name)
        if path is None:
            missing.append(name)
            continue

        match = ARTIFACT_PATTERN.match(name)
        if match is None:
            continue

        payload = json.loads(path.read_text(encoding="utf-8"))
        sharding = payload.get("sharding") or {}
        records.append(
            {
                "artifact": name,
                "dataset": match.group("dataset"),
                "family": match.group("family"),
                "batch": int(match.group("batch")),
                "backend": payload.get("backend"),
                "num_images": payload.get("num_images"),
                "num_padded_images": payload.get("num_padded_images"),
                "warmup_steps": payload.get("warmup_steps"),
                "benchmark_steps": payload.get("benchmark_steps"),
                "mean_step_time_sec": payload.get("mean_step_time_sec"),
                "throughput_images_per_sec": payload.get("throughput_images_per_sec"),
                "visible_devices": len(payload.get("devices") or []),
                "device_kind": (payload.get("devices") or [{}])[0].get("device_kind"),
                "sharding_mode": sharding.get("mode"),
                "mesh": clean_mesh(sharding.get("mesh_shape")),
                "mesh_device_count": sharding.get("mesh_device_count"),
                "per_device_batch_size": sharding.get("per_device_batch_size"),
                "image_partition_spec": sharding.get("image_partition_spec"),
                "logits_partition_spec": sharding.get("logits_partition_spec"),
                "jit_sharding_level": clean_jit_level(
                    sharding.get("jit_sharding_level")
                ),
                "fallback": clean_fallback(sharding.get("fallback")),
            }
        )

    records.sort(key=lambda row: (row["dataset"], row["family"], row["batch"]))
    return records, missing


def format_value(value: Any, column: str) -> str:
    if value is None:
        return ""
    if isinstance(value, float):
        if not math.isfinite(value):
            return str(value)
        if "mean step" in column:
            return f"{value:.6f}"
        if "img/s" in column:
            return f"{value:.1f}"
        if "throughput" in column and column != "throughput img/s":
            return f"{value:.3f}x"
        return f"{value:.3f}"
    return str(value)


def markdown_table(rows: list[dict[str, Any]], columns: list[str]) -> str:
    header = "| " + " | ".join(columns) + " |"
    separator = "| " + " | ".join("---" for _ in columns) + " |"
    body = [
        "| "
        + " | ".join(format_value(row.get(column), column) for column in columns)
        + " |"
        for row in rows
    ]
    return "\n".join([header, separator, *body])


def show_missing_artifacts(missing: list[str]) -> None:
    if missing:
        display(
            Markdown(
                "**Missing raw JSON artifacts:**\n\n"
                + "\n".join(f"- `{name}`" for name in missing)
                + "\n\nThe tables below will use the artifacts that are available."
            )
        )

## 4. Inference Evidence I: Run Families and Resolved Layout

The raw JSON artifacts record the resolved sharding layout and the timing results. The notebook summarizes sharding metadata by run family rather than listing every JSON file, because the important layout fields are almost identical within a family.

The fields that vary by artifact are the global batch size, per-device batch size, image count, padding count, and timing. The table below separates those varying fields from the stable layout fields.

In [ ]:
records, missing = load_artifact_records()
show_missing_artifacts(missing)

family_rows: list[dict[str, Any]] = []
for dataset in ("val256", "valfull"):
    for family in ("single_v6e1", "sharded_v6e8"):
        rows = [
            row
            for row in records
            if row["dataset"] == dataset and row["family"] == family
        ]
        if not rows:
            continue
        first = rows[0]
        if family == "single_v6e1":
            per_device_batches = "same as global batch"
        else:
            per_device_batches = ", ".join(
                str(row["per_device_batch_size"]) for row in rows
            )
        family_rows.append(
            {
                "dataset": dataset,
                "family": sharding_label(family),
                "batches": ", ".join(str(row["batch"]) for row in rows),
                "images": first["num_images"],
                "warmup": first["warmup_steps"],
                "steps": first["benchmark_steps"],
                "visible devices": first["visible_devices"],
                "sharding mode": first["sharding_mode"],
                "per-device batches": per_device_batches,
            }
        )

display(
    Markdown(
        "### Run-family summary\n\n"
        + markdown_table(
            family_rows,
            [
                "dataset",
                "family",
                "batches",
                "images",
                "warmup",
                "steps",
                "visible devices",
                "sharding mode",
                "per-device batches",
            ],
        )
    )
)

layout_rows: list[dict[str, Any]] = []
for family in ("single_v6e1", "sharded_v6e8"):
    rows = [row for row in records if row["family"] == family]
    if not rows:
        continue
    first = rows[0]
    layout_rows.append(
        {
            "family": sharding_label(family),
            "mesh": first["mesh"],
            "image layout": first["image_partition_spec"]
            or "replicated/unsharded input",
            "logits layout": first["logits_partition_spec"] or "ordinary output",
            "jit level": first["jit_sharding_level"],
            "fallback": first["fallback"],
        }
    )

display(
    Markdown(
        "### Resolved layout summary\n\n"
        + markdown_table(
            layout_rows,
            [
                "family",
                "mesh",
                "image layout",
                "logits layout",
                "jit level",
                "fallback",
            ],
        )
    )
)

## 5. Inference Evidence II: Single-Device vs Sharded Multi-Device Timing

The timing comparison asks how the sharded layout behaves as the global batch size grows.

```text
multi/single throughput = sharded v6e-8 throughput / single v6e-1 throughput
```

A value below `1x` means the single-device run is faster for that batch size. A value above `1x` means the sharded multi-device run is faster for that batch size.

In [ ]:
def comparison_rows(dataset: str) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    batches = sorted({row["batch"] for row in records if row["dataset"] == dataset})
    for batch in batches:
        single = next(
            (
                row
                for row in records
                if row["dataset"] == dataset
                and row["family"] == "single_v6e1"
                and row["batch"] == batch
            ),
            None,
        )
        sharded = next(
            (
                row
                for row in records
                if row["dataset"] == dataset
                and row["family"] == "sharded_v6e8"
                and row["batch"] == batch
            ),
            None,
        )
        if single is None or sharded is None:
            continue

        ratio = (
            sharded["throughput_images_per_sec"] / single["throughput_images_per_sec"]
        )
        rows.append(
            {
                "batch": batch,
                "per-device batch": sharded["per_device_batch_size"],
                "images": single["num_images"],
                "padded": single["num_padded_images"],
                "single mean step (s)": single["mean_step_time_sec"],
                "multi mean step (s)": sharded["mean_step_time_sec"],
                "single img/s": single["throughput_images_per_sec"],
                "multi img/s": sharded["throughput_images_per_sec"],
                "multi/single throughput": ratio,
            }
        )
    return rows


comparison_columns = [
    "batch",
    "per-device batch",
    "images",
    "padded",
    "single mean step (s)",
    "multi mean step (s)",
    "single img/s",
    "multi img/s",
    "multi/single throughput",
]

for dataset in ("val256", "valfull"):
    rows = comparison_rows(dataset)
    display(
        Markdown(
            f"### {dataset}: single-device vs sharded multi-device\n\n"
            + markdown_table(rows, comparison_columns)
        )
    )

In [ ]:
# Optional visualization. The tables above are the primary evidence; the plots
# provide a visual check of the same crossover pattern.
try:
    import matplotlib.pyplot as plt
except ImportError as exc:
    display(
        Markdown(
            f"matplotlib is unavailable, so the throughput plots are skipped: `{exc}`"
        )
    )
else:
    for dataset in ("val256", "valfull"):
        fig, ax = plt.subplots(figsize=(7, 4))
        for family, label in (
            ("single_v6e1", "single v6e-1"),
            ("sharded_v6e8", "sharded v6e-8"),
        ):
            rows = sorted(
                [
                    row
                    for row in records
                    if row["dataset"] == dataset and row["family"] == family
                ],
                key=lambda row: row["batch"],
            )
            if not rows:
                continue
            ax.plot(
                [row["batch"] for row in rows],
                [row["throughput_images_per_sec"] for row in rows],
                marker="o",
                label=label,
            )
        ax.set_xscale("log", base=2)
        ax.set_xlabel("Global batch size")
        ax.set_ylabel("Throughput (images/sec)")
        ax.set_title(f"{dataset}: ViT inference throughput")
        ax.legend()
        ax.grid(True, which="both", axis="both")
        plt.show()

    display(
        Markdown(
            "The plots visualize the same crossover shown in the tables: "
            "the single-device curve dominates at small batches, while the sharded "
            "v6e-8 curve overtakes at larger global batches."
        )
    )

## 6. Interpreting the Timing Results

The timing results show a crossover pattern.

For `val256`, the sharded `v6e-8` run is slower than the single-device `v6e-1` run at global batch sizes 8, 16, and 64. At batch size 256, the sharded run becomes slightly faster. This matches the per-device batch sizes: at global batch 8, each of the eight devices receives only one image; at global batch 256, each device receives 32 images.

For the full validation manifest, the pattern is stronger. The crossover occurs between batch size 64 and batch size 256 in these artifacts. At batch size 8, the sharded run has only one image per device and its throughput is far below the single-device run. At batch size 256, the two profiles are near break-even. At batch size 1024, the sharded run reaches much higher throughput because each device receives a larger local batch and the fixed multi-device overhead is better amortized.

The key interpretation is:

```text
Explicit batch-axis sharding is useful when the global batch is large enough to provide meaningful work to every device.
```

For small batches, the single-device TPU profile is better for this workload. For large batches, the eight-device sharded profile can process more images per unit time. The measured advantage at large batch size is an **inference-throughput** advantage. It does not imply better single-image latency, better training speed, or better accuracy.

## 7. Fine-Tuning Batch Inputs: Code Support Only

The fine-tuning script applies the same batch-axis convention to training-shaped inputs:

| Input field | Sharding |
| --- | --- |
| `pixel_values` | `PartitionSpec("data", None, None, None)` |
| `labels` | `PartitionSpec("data")` |
| `mask` | `PartitionSpec("data")` |

The train step leaves `head_params` and `optimizer_state` outside the explicit sharding contract, while the batch-shaped arrays receive explicit shardings:

```python
in_shardings=(
    None,            # head_params
    None,            # optimizer_state
    image_sharding,  # pixel_values
    label_sharding,  # labels
    mask_sharding,   # mask
)
```

This section is included to show consistent use of the batch-axis layout beyond inference. The measured evidence in this report remains inference-only.

## 8. Alternative JAX Sharding Methods

Demo 2 uses a conservative and inspectable subset of JAX sharding: explicit data sharding at the input/output boundary of a ViT inference step. Other JAX/Flax methods become useful when the bottleneck shifts from batch throughput to model size, activation memory, optimizer state, or custom communication.

| Method | Useful when | Minimal example | Relationship to Demo 2 |
| --- | --- | --- | --- |
| Compiler-guided layout constraints | Many tensors exist and most layouts can be left to the compiler, with selected constraints. | `with_sharding_constraint(y, P("data", "model"))` | Demo 2 uses explicit artifact metadata, so the image/logits layouts are stated directly. |
| Explicit sharding at `jit` boundaries | A small number of input/output layouts should be inspectable and reproducible. | `jax.jit(step, in_shardings=(None, image_sharding))` | This is the implemented method. |
| Manual SPMD with `shard_map` | The algorithm requires explicit per-shard computation and collectives. | `shard_map(step, mesh=mesh, in_specs=P("data", None), ...)` | ViT inference here does not require custom collectives. |
| Flax parameter sharding | Model weights or optimizer state consume too much per-device memory when replicated. | `with_partitioning(init, (None, "model"))` | The successful single-device inference artifacts show that model-state memory is not the limiting factor in this report. |
| Flax activation sharding | Intermediate activations dominate memory or communication. | `with_sharding_constraint(y, P("data", "model"))` | Demo 2 does not shard inside ViT blocks. |
| Orbax sharded checkpointing | A distributed training state must be saved and restored in its sharded layout. | save/restore a PyTree with sharded `jax.Array` leaves | Demo 2 checkpoint evidence is classifier-head oriented, not full sharded model-state restore. |

Parameter sharding is best understood through a memory budget. With bf16 or fp16 weights only, one billion parameters require about 2 GB for inference weights. During Adam-like training, the budget is much larger because parameters, gradients, and optimizer accumulators can require roughly 12 to 16 bytes per parameter before activations are counted. On a 32 GB TPU v6e chip, inference for a moderately sized model may still fit comfortably on one device, while multi-billion-parameter training can become tight or infeasible without parameter, optimizer-state, activation, or checkpoint sharding.

These numbers are practical estimates rather than fixed thresholds. The actual decision depends on dtype, optimizer, activation size, batch size, sequence length, rematerialization, and checkpoint strategy. For Demo 2, the inference artifacts already show that the ViT model can run on a single `v6e-1` device. The report therefore studies throughput scaling from input-batch sharding, not model-state memory pressure.

## 9. Summary

The Demo 2 sharding extension demonstrates a focused use of JAX sharding:

1. the image batch uses `PartitionSpec("data", None, None, None)` over a one-dimensional `data` mesh;
2. image batches are placed with `jax.device_put` onto a `NamedSharding`;
3. the ViT inference step is compiled with `jax.jit(in_shardings=..., out_shardings=...)`;
4. raw JSON artifacts record the resolved layout and timing behavior;
5. the single-device versus sharded multi-device comparison shows that batch-axis sharding becomes useful when the global batch is large enough to provide meaningful work to every device.

The main lesson is that JAX sharding is a layout contract, not simply a speed switch. In this project, making that contract explicit produced reportable metadata and a clear inference-throughput comparison between single-device and sharded multi-device TPU runs.

## References

- JAX documentation: [Distributed arrays and automatic parallelization](https://docs.jax.dev/en/latest/parallel.html)
- JAX API reference: [`jax.sharding`](https://docs.jax.dev/en/latest/jax.sharding.html)
- JAX API reference: [`jax.jit`](https://docs.jax.dev/en/latest/_autosummary/jax.jit.html)
- JAX API reference: [`jax.device_put`](https://docs.jax.dev/en/latest/_autosummary/jax.device_put.html)
- JAX API reference: [`jax.lax.with_sharding_constraint`](https://docs.jax.dev/en/latest/_autosummary/jax.lax.with_sharding_constraint.html)
- JAX guide: [Manual parallelism with `shard_map`](https://docs.jax.dev/en/latest/notebooks/shard_map.html)
- Flax NNX guide: [Scale up on multiple devices](https://flax.readthedocs.io/en/latest/guides/flax_gspmd.html)
- Flax Linen guide: [Scale up Flax Modules on multiple devices](https://flax-linen.readthedocs.io/en/latest/guides/parallel_training/flax_on_pjit.html)
- Orbax guide: [Checkpointing with Orbax](https://orbax.readthedocs.io/en/latest/guides/checkpoint/orbax_checkpoint_101.html)
- Google Cloud documentation: [TPU v6e architecture and specifications](https://cloud.google.com/tpu/docs/v6e)